# BZT-SHIS v2 — Criticality-Aware, Ledger-Anchored Zero Trust for Clinical IoT

**One-sentence claim.** A zero-trust access framework for hospital IoT in which (1) device identity, revocation and
replay protection are enforced by a *permissioned ledger gate before any model runs*, (2) a continuous, time-decayed
trust history is kept per device, (3) decisions are *cost-sensitive and criticality-aware* (allow / restrict / deny),
(4) every decision carries a hashed explanation that is anchored on the ledger, and (5) models are trained per hospital and
combined by a one-shot federated ensemble whose updates are also ledger-recorded.

**What changed from v1 (the original notebook)**

| Issue in v1 | Fix in v2 |
|---|---|
| Metrics computed on the whole dataset, including training rows | Time-based train / calibration / test split + forward-chaining folds (mean ± std) |
| Blockchain only logged decisions afterwards | Ledger *gate* decides before the model; revocations and replay cache are ledger state |
| Weak baselines (55.9 % logistic regression, "predict all legitimate") | RF, HistGradientBoosting, MLP, tuned per-device-type perimeter rule, plus a stepwise ablation |
| Binary grant / deny at 0.5 | Calibrated probabilities + expected-cost policy with per-device criticality |
| "Privacy-preserving" in the title, nothing implemented | Per-hospital training, one-shot federated ensemble, hashed DIDs on the ledger (no formal DP — see limitations) |
| Fake nonce "mining", fresh AES key per message, no decrypt | Quorum-signed (4 validators, 3 needed) Merkle-batched ledger, X25519 → HKDF → AES-256-GCM with round-trip + tamper tests |
| "Network overhead 100 %" bookkeeping artefact | Overhead measured in bytes per message |

> **Data note (read first).** Network behaviour is **real TON_IoT** (`Train_Test_Network.csv`: normal + 9 native attack classes,
> 20,569 exact duplicate rows removed). TON_IoT has no identity layer, so `build_toniot_sessions.py` adds one: device DIDs, a registry
> with revocations, signatures, nonces, hospitals (derived from the victim IP) and device criticality (assigned at random, so it drives the
> *policy* layer, not the classifier). Four identity attacks (spoofing, replay, unregistered DID, revoked device) are **injected** at known rates
> with a real normal flow as their traffic (`source = injected_identity`). Raw `src_ip` is *not* a feature (it would leak the label); it is kept only for the
> hold-out check in section 7b. The Train_Test file has **no timestamps**, so time order is synthetic and the "chronological" split behaves like a random split,
> and the ledger-history features have no real timeline behind them. Set `DATASET = "synthetic"` to use the fully generated set instead.

## 1. Setup

In [ ]:
import os, sys, time, json, hmac, copy, pickle, hashlib, warnings
from collections import defaultdict, deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
DATA_DIR, FIG_DIR, RES_DIR = ".", "figures", "results"
for d in (FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)
N_TREES = 120
DATASET = "toniot"      # "toniot" = real TON_IoT flows + injected identity layer | "synthetic" = fully generated
SESSIONS_FILE = {"toniot": "toniot_zt_sessions.csv", "synthetic": "healthiot_zt_sessions.csv"}[DATASET]
REGISTRY_FILE = {"toniot": "device_registry_toniot.csv", "synthetic": "device_registry.csv"}[DATASET]

COL = {"base": "#8A93A6", "full": "#1B6CA8", "accent": "#D9822B", "bad": "#C0392B", "good": "#2E8B57", "soft": "#A9C6E0"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 10})

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight")
    plt.show()

## 2. Data — real TON_IoT flows + injected identity layer
If the two prepared CSVs are missing, the notebook builds them from `Train_Test_Network.csv` with `build_toniot_sessions.py`.

In [ ]:
def load_data():
    s = os.path.join(DATA_DIR, SESSIONS_FILE)
    r = os.path.join(DATA_DIR, REGISTRY_FILE)
    if not (os.path.exists(s) and os.path.exists(r)):
        if DATASET != "synthetic":
            src = os.path.join(DATA_DIR, "Train_Test_Network.csv")
            assert os.path.exists(src), f"upload {SESSIONS_FILE} + {REGISTRY_FILE}, or Train_Test_Network.csv with build_toniot_sessions.py"
            from build_toniot_sessions import build
            d, reg = build(src, 40_000, seed=SEED)
            d.to_csv(s, index=False); reg.to_csv(r, index=False)
            return d, reg
        from generate_dataset import generate
        d, reg = generate(seed=SEED)
        d.to_csv(s, index=False); reg.to_csv(r, index=False)
    return pd.read_csv(s), pd.read_csv(r)

df, reg = load_data()
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
TYPE_CODE = {t: i for i, t in enumerate(sorted(df.device_type.unique()))}
df["type_code"] = df.device_type.map(TYPE_CODE)
y = df.label.values
crit = df.criticality.values.astype(int)
sens = df.data_sensitivity.values.astype(int)
hosp = df.hospital_id.values.astype(int)

n = len(df)
tr  = np.arange(0, int(.60 * n))              # fit
cal = np.arange(int(.60 * n), int(.75 * n))   # calibration (isotonic) - never used for fitting trees
te  = np.arange(int(.75 * n), n)              # untouched test period (latest 25 %)
print(f"sessions={n:,}  devices={len(reg)}  attack rate={y.mean():.1%}")
print(f"train={len(tr):,}  calibration={len(cal):,}  test={len(te):,}  (chronological by timestamp" + ("; TON_IoT timestamps are synthetic, so this behaves like a random split)" if DATASET == "toniot" else ")"))
df.head(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
df.attack_type.value_counts().sort_values().plot.barh(ax=ax[0], color=COL["full"])
ax[0].set_title("Sessions by type"); ax[0].set_xlabel("count")
(df.groupby("device_type").label.mean().sort_values() * 100).plot.barh(ax=ax[1], color=COL["accent"])
ax[1].set_title("Attack share by device type (%)")
(df.groupby("hospital_id").label.mean() * 100).plot.bar(ax=ax[2], color=COL["base"])
ax[2].set_title("Attack share by hospital (%) - non-IID"); ax[2].set_xlabel("hospital")
savefig("fig0_dataset_overview")
print(df.groupby("criticality").size().rename("sessions per criticality (4 = life-critical)").to_string())

## 3. Permissioned ledger (simulated)

* 4 validators, **3 signatures needed** (tolerates 1 faulty validator, the usual 3f+1 rule). Signatures are HMAC-SHA256
  stand-ins for real validator signatures.
* Transactions are batched into blocks with a **Merkle root**, so a single decision can be audited in O(log n) without scanning the chain.
* World state holds the **device registry** (DID → firmware hash, revoked flag) and **model-update records**.
* It is a *simulation of consensus*, not a distributed system. Say "simulated permissioned ledger" in the paper.

In [ ]:
def sha(b):
    return hashlib.sha256(b).hexdigest()

def tx_hash(tx):
    return sha(json.dumps(tx, sort_keys=True, separators=(",", ":")).encode())

def merkle_root(hashes):
    if not hashes:
        return sha(b"")
    lvl = list(hashes)
    while len(lvl) > 1:
        if len(lvl) % 2:
            lvl.append(lvl[-1])
        lvl = [sha((lvl[i] + lvl[i + 1]).encode()) for i in range(0, len(lvl), 2)]
    return lvl[0]

def merkle_proof(hashes, idx):
    proof, lvl = [], list(hashes)
    while len(lvl) > 1:
        if len(lvl) % 2:
            lvl.append(lvl[-1])
        sib = idx ^ 1
        proof.append((lvl[sib], sib < idx))
        lvl = [sha((lvl[i] + lvl[i + 1]).encode()) for i in range(0, len(lvl), 2)]
        idx //= 2
    return proof

def verify_proof(leaf, proof, root):
    h = leaf
    for sib, sib_is_left in proof:
        h = sha((sib + h).encode()) if sib_is_left else sha((h + sib).encode())
    return h == root


class PermissionedLedger:
    def __init__(self, n_validators=4, quorum=3, seed=SEED):
        self.keys = [hashlib.sha256(f"validator-{i}-{seed}".encode()).digest() for i in range(n_validators)]
        self.quorum = quorum
        self.chain = []
        self.state = {"registry": {}, "models": []}

    def _sign(self, i, h):
        return hmac.new(self.keys[i], h.encode(), "sha256").hexdigest()

    @staticmethod
    def _hdr(index, ts, prev, root, proposer):
        return sha(f"{index}|{ts}|{prev}|{root}|{proposer}".encode())

    def _apply(self, t):
        k = t.get("type")
        if k == "REGISTER":
            self.state["registry"][t["did"]] = {"fw": t["fw"], "type": t["device_type"], "revoked": False}
        elif k == "REVOKE":
            self.state["registry"][t["did"]]["revoked"] = True
        elif k == "MODEL_UPDATE":
            self.state["models"].append(t)

    def append(self, txs, ts, signers=None):
        signers = list(range(len(self.keys))) if signers is None else list(signers)
        if len(signers) < self.quorum:               # no quorum -> block rejected, state untouched
            return None
        idx = len(self.chain)
        prev = self.chain[-1]["hash"] if self.chain else "0" * 64
        hs = [tx_hash(t) for t in txs]
        root = merkle_root(hs)
        prop = idx % len(self.keys)
        h = self._hdr(idx, int(ts), prev, root, prop)
        blk = dict(index=idx, ts=int(ts), prev=prev, root=root, proposer=prop, hash=h,
                   txs=txs, tx_hashes=hs, sigs={i: self._sign(i, h) for i in signers})
        self.chain.append(blk)
        for t in txs:
            self._apply(t)
        return blk

    def _check_block(self, b, prev_hash):
        hs = [tx_hash(t) for t in b["txs"]]
        if hs != b["tx_hashes"]:
            return "transaction content does not match its recorded hash"
        if merkle_root(hs) != b["root"]:
            return "Merkle root mismatch"
        if self._hdr(b["index"], b["ts"], b["prev"], b["root"], b["proposer"]) != b["hash"]:
            return "block header hash mismatch"
        if prev_hash is not None and b["prev"] != prev_hash:
            return "broken link to previous block"
        good = sum(1 for i, s in b["sigs"].items() if hmac.compare_digest(s, self._sign(i, b["hash"])))
        if good < self.quorum:
            return f"only {good} valid validator signatures (< quorum {self.quorum})"
        return None

    def verify(self):
        prev = None
        for b in self.chain:
            err = self._check_block(b, prev)
            if err:
                return False, b["index"], err
            prev = b["hash"]
        return True, None, "ok"

    def audit_tx(self, block_idx, tx_idx):
        # O(log n) audit of ONE decision: Merkle proof + header + quorum signatures + link to previous block
        b = self.chain[block_idx]
        leaf = tx_hash(b["txs"][tx_idx])
        if not verify_proof(leaf, merkle_proof(b["tx_hashes"], tx_idx), b["root"]):
            return False
        prev_hash = self.chain[block_idx - 1]["hash"] if block_idx else None
        if prev_hash is not None and b["prev"] != prev_hash:
            return False
        if self._hdr(b["index"], b["ts"], b["prev"], b["root"], b["proposer"]) != b["hash"]:
            return False
        return sum(1 for i, s in b["sigs"].items() if hmac.compare_digest(s, self._sign(i, b["hash"]))) >= self.quorum


# ---- seed the ledger with the registry (genesis) ----
BASE_REGISTRY = {r.did: {"fw": r.fw_hash, "type": r.device_type, "revoked": False} for r in reg.itertuples()}
REV_EVENTS = sorted((float(r.revoked_at), r.did) for r in reg.itertuples() if pd.notna(r.revoked_at))

def new_seeded_ledger():
    L = PermissionedLedger()
    items = list(BASE_REGISTRY.items())
    for i in range(0, len(items), 100):
        L.append([{"type": "REGISTER", "did": d, "fw": v["fw"], "device_type": v["type"]} for d, v in items[i:i + 100]], ts=0)
    return L

ledger = new_seeded_ledger()
print(f"ledger seeded: {len(ledger.chain)} blocks, {len(ledger.state['registry'])} registered devices,"
      f" {len(REV_EVENTS)} devices will be revoked during the observation window")

## 4. Ledger gate + continuous trust history (streaming, strictly causal)

For every session, in time order, the gate reads **ledger state** and applies hard checks:

`UNREGISTERED_DID` → `REVOKED_DID` → `BAD_SIGNATURE` → `REPLAY_NONCE`

A firmware-hash mismatch is a *soft* signal (legitimate updates lag behind registration), so it becomes a model feature.
Per-DID history is kept as a **time-decayed risk average** (`ema_prev`, half-life 6 h, decaying towards a neutral prior),
the number of gate failures in the last 24 h, the gap since the previous session and the request rate in the last hour.
Only information from *earlier* sessions is used.

In [ ]:
HALF_LIFE_MIN, PRIOR, ALPHA = 360.0, 0.2, 0.3

def stream(df, p_net, ledger=None):
    # Causal pass over sessions. Returns gate decision + ledger-derived history features.
    reg_state = ledger.state["registry"] if ledger is not None else {d: dict(v) for d, v in BASE_REGISTRY.items()}
    ts, did, fw = df.timestamp.values, df.claimed_did.values, df.presented_fw_hash.values
    sig, nonce = df.sig_ok.values, df.nonce.values
    N = len(df)
    gate = np.full(N, "", dtype=object)
    fwm = np.zeros(N); ema_prev = np.zeros(N); fails24 = np.zeros(N); gap = np.zeros(N); rate1h = np.zeros(N)
    seen, ema, last = defaultdict(set), {}, {}
    fails, recent = defaultdict(deque), defaultdict(deque)
    ptr = 0
    for k in range(N):
        t, d = ts[k], did[k]
        while ptr < len(REV_EVENTS) and REV_EVENTS[ptr][0] <= t:          # revocations become ledger state
            rt, rd = REV_EVENTS[ptr]; ptr += 1
            if ledger is not None:
                ledger.append([{"type": "REVOKE", "did": rd}], ts=rt)
            else:
                reg_state[rd]["revoked"] = True
        entry = reg_state.get(d)
        if d in last:
            g = (t - last[d]) / 60.0
            e = PRIOR + (ema[d] - PRIOR) * 0.5 ** (g / HALF_LIFE_MIN)     # trust decays towards the prior over time
        else:
            g, e = 10080.0, PRIOR
        fq, rq = fails[d], recent[d]
        while fq and fq[0] < t - 86400: fq.popleft()
        while rq and rq[0] < t - 3600: rq.popleft()
        ema_prev[k], fails24[k], gap[k], rate1h[k] = e, len(fq), g, len(rq)
        if entry is None:            reason = "UNREGISTERED_DID"
        elif entry["revoked"]:       reason = "REVOKED_DID"
        elif not sig[k]:             reason = "BAD_SIGNATURE"
        elif nonce[k] in seen[d]:    reason = "REPLAY_NONCE"
        else:                        reason = ""
        seen[d].add(nonce[k])
        gate[k] = reason
        fwm[k] = 1.0 if (entry is None or fw[k] != entry["fw"]) else 0.0
        risk = max(p_net[k], 0.9) if reason else p_net[k]
        ema[d] = ALPHA * risk + (1 - ALPHA) * e
        last[d] = t; rq.append(t)
        if reason: fq.append(t)
    return dict(gate=gate, fw_mismatch=fwm, ema_prev=ema_prev, fails24=fails24, gap_min=gap, rate1h=rate1h)

# quick demo: gate alone (no ML) on the whole stream
t0 = time.perf_counter(); H_demo = stream(df, np.full(len(df), PRIOR)); dt = time.perf_counter() - t0
print(f"gate + history over {len(df):,} sessions: {dt:.2f}s  ({dt / len(df) * 1e6:.1f} us/session)")
print(pd.crosstab(df.attack_type, np.where(H_demo['gate'] == '', 'passes gate', H_demo['gate'])).to_string())

## 5. Two-stage model pipeline

* **Stage 1** – Random Forest on network + clinical-context features → `p_net` (out-of-fold on the training period, so
  that downstream models do not see over-confident in-sample scores). This is also the **ML-only baseline**.
* **Stage 2** – Random Forest on `network + context + p_net + ledger-history` features, trained *only on sessions that pass the gate*
  (the gate has already removed the trivially detectable identity violations). Probabilities are then **isotonic-calibrated**
  on the calibration window, which is a prerequisite for cost-based decisions.

In [ ]:
NET_ALL = ["duration_s", "src_bytes", "dst_bytes", "src_pkts", "dst_pkts", "src_ip_bytes", "dst_ip_bytes", "missed_bytes",
           "conn_state_code", "proto_code", "has_http", "has_ssl", "has_dns", "req_per_min", "data_volume_kb",
           "dst_port", "service_code", "http_request_body_len", "http_response_body_len", "http_status_code", "dns_qtype",
           "dns_rcode", "ssl_established_flag", "weird_flag", "dns_flag",
           "rtt_ms", "jitter_ms", "payload_entropy", "vitals_zscore", "hour_of_day"]
NET = [c for c in NET_ALL if c in df.columns]            # works for both datasets: uses whichever columns exist
SKEW_SET = {"duration_s", "src_bytes", "dst_bytes", "src_pkts", "dst_pkts", "src_ip_bytes", "dst_ip_bytes", "missed_bytes",
            "req_per_min", "data_volume_kb", "rtt_ms", "jitter_ms", "http_request_body_len", "http_response_body_len"}
print("network features used:", NET)
CTX  = ["criticality", "data_sensitivity", "type_code"]
HIST = ["p_net", "ema_prev", "fails24", "gap_min", "rate1h", "fw_mismatch"]
F1_COLS, F2_COLS = NET + CTX, NET + CTX + HIST

def rf(n=N_TREES, **kw):
    return RandomForestClassifier(n_estimators=n, min_samples_leaf=2, n_jobs=-1, random_state=SEED, **kw)

def build_X2(df, p_net, H):
    return np.column_stack([df[NET + CTX].values, p_net, H["ema_prev"], H["fails24"], H["gap_min"], H["rate1h"], H["fw_mismatch"]])

LOGCOLS = [c for c in NET if c in SKEW_SET and c != "data_volume_kb"]
LOG_IDX = [F1_COLS.index(c) for c in LOGCOLS]
J_SB, J_DB, J_VOL = F1_COLS.index("src_bytes"), F1_COLS.index("dst_bytes"), F1_COLS.index("data_volume_kb")
INT_COLS = [c for c in ["src_bytes", "dst_bytes", "src_pkts", "dst_pkts", "src_ip_bytes", "dst_ip_bytes", "missed_bytes",
                        "http_request_body_len", "http_response_body_len"] if c in F1_COLS]
INT_IDX = [F1_COLS.index(c) for c in INT_COLS]

SIZE_IDX = [F1_COLS.index(c) for c in (["src_ip_bytes", "dst_ip_bytes"] if "src_ip_bytes" in F1_COLS else ["src_bytes", "dst_bytes"])]
TIME_IDX = [j for j in LOG_IDX if j not in INT_IDX]            # durations, rtt, ... (continuous)

def compute_med(X1, yy, tr):
    base = X1[tr][yy[tr] == 0]
    med = {j: np.median(base[:, j]) for j in LOG_IDX}
    med["size"] = np.median(base[:, SIZE_IDX].sum(axis=1))
    return med

def perturb_X1(X, eps, med):
    # Evasion model: the attacker throttles the flow towards the size of a median normal flow. All count features (bytes, packets, IP bytes,
    # HTTP body lengths) are scaled by ONE common factor, so their relationships (bytes per packet, header overhead) stay consistent,
    # and then rounded to integers. Continuous timing features move towards the normal median. Protocol / service / state are NOT changed.
    Z = X.astype(float).copy()
    size = Z[:, SIZE_IDX].sum(axis=1)
    f = ((med["size"] + 1.0) / (size + 1.0)) ** eps
    for j in INT_IDX:
        Z[:, j] = np.round(Z[:, j] * f)
    for j in TIME_IDX:
        Z[:, j] = np.expm1(eps * np.log1p(med[j]) + (1 - eps) * np.log1p(Z[:, j]))
    Z[:, J_VOL] = (Z[:, J_SB] + Z[:, J_DB]) / 1024.0
    return Z

def fit_pipeline(df, tr, cal, robust=False, K=2):
    # robust=True adds evasion-aware training: K perturbed copies of every training NETWORK/DATA attack (eps ~ U(0.2, 1)) are added to
    # stage 1, and to stage 2 with p_net recomputed out-of-fold on the perturbed features. History features are kept as they were.
    X1, yy = df[F1_COLS].values, df.label.values
    rng_a = np.random.default_rng(SEED)
    med = compute_med(X1, yy, tr)
    isnet = np.isin(df.attack_family.values, ["network", "data"])     # only attackers that rely on their traffic shape can evade this way
    def with_aug(Xs, ys, nets):
        if not robust: return Xs, ys
        A = Xs[(ys == 1) & nets]
        ex = np.vstack([perturb_X1(A, rng_a.uniform(.2, 1.0, len(A)), med) for _ in range(K)])
        return np.vstack([Xs, ex]), np.r_[ys, np.ones(len(ex), int)]
    m_net = rf().fit(*with_aug(X1[tr], yy[tr], isnet[tr]))
    oof = np.zeros(len(tr)); XP = np.zeros((K, len(tr), X1.shape[1])); oof_aug = np.zeros((K, len(tr)))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, yy[tr]):
        mf = rf(80).fit(*with_aug(X1[tr][a], yy[tr][a], isnet[tr][a]))
        oof[b] = mf.predict_proba(X1[tr][b])[:, 1]
        if robust:
            for k in range(K):
                XP[k, b] = perturb_X1(X1[tr][b], rng_a.uniform(.2, 1.0, len(b)), med)
                oof_aug[k, b] = mf.predict_proba(XP[k, b])[:, 1]
    p_net = np.zeros(len(df)); rest = np.setdiff1d(np.arange(len(df)), tr)
    p_net[tr] = oof; p_net[rest] = m_net.predict_proba(X1[rest])[:, 1]
    H = stream(df, p_net)
    X2 = build_X2(df, p_net, H); ok = H["gate"] == ""
    trg, calg = tr[ok[tr]], cal[ok[cal]]
    Xs2, ys2 = X2[trg], yy[trg]
    if robust:                                         # perturbed attack copies for stage 2 (gate-passed training attacks only)
        pos = np.where(ok[tr] & (yy[tr] == 1) & isnet[tr])[0]
        ex = []
        for k in range(K):
            R = X2[tr][pos].copy(); R[:, :len(F1_COLS)] = XP[k, pos]; R[:, len(F1_COLS)] = oof_aug[k, pos]; ex.append(R)
        ex = np.vstack(ex); Xs2, ys2 = np.vstack([Xs2, ex]), np.r_[ys2, np.ones(len(ex), int)]
    m2 = rf().fit(Xs2, ys2)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(m2.predict_proba(X2[calg])[:, 1], yy[calg])
    return dict(m_net=m_net, oof=oof, m2=m2, iso=iso, tr=tr, robust=robust)

def apply_pipeline(b, df, te, ledger=None):
    X1, tr_ = df[F1_COLS].values, b["tr"]
    p_net = np.zeros(len(df)); rest = np.setdiff1d(np.arange(len(df)), tr_)
    p_net[tr_] = b["oof"]; p_net[rest] = b["m_net"].predict_proba(X1[rest])[:, 1]
    H = stream(df, p_net, ledger)
    X2 = build_X2(df, p_net, H); ok = H["gate"] == ""
    p2 = np.ones(len(df)); pc = np.ones(len(df))        # gate failures: attack probability 1
    ti = te[ok[te]]
    p2[ti] = b["m2"].predict_proba(X2[ti])[:, 1]
    pc[ti] = b["iso"].predict(p2[ti])
    return dict(p_net=p_net, H=H, X2=X2, p2=p2, pc=pc, ok=ok)

## 6. Criticality-aware, cost-sensitive policy

Costs are *design parameters* (state and justify them in the paper; run a sensitivity analysis).

* Cost of wrongly **allowing** an attack: `5 + 3 × data_sensitivity` (breach impact).
* Cost of wrongly **denying** a legitimate device: 1 / 2 / 5 / 10 for criticality 1 / 2 / 3 / 4 — cutting off a ventilator is far worse than cutting off an environment sensor.
* **Restrict** (read-only / quarantine VLAN / clinician alert) costs 30 % of the corresponding full error either way.

With calibrated `p = P(attack)` the minimum-expected-cost action is chosen per session, so life-critical devices need much
stronger evidence before being denied and are *restricted* instead.

In [ ]:
CFR = np.array([0, 1, 2, 5, 10.0])
RESTRICT_F = 0.3
def c_fa(sens_): return 5 + 3 * sens_

def policy_tier(p, crit_, sens_):
    cfr, cfa = CFR[crit_], c_fa(sens_)
    allow = p * cfa
    restr = (1 - p) * RESTRICT_F * cfr + p * RESTRICT_F * cfa
    deny = (1 - p) * cfr
    return np.argmin(np.vstack([allow, restr, deny]), axis=0)      # 0 allow, 1 restrict, 2 deny

def session_cost(y_, tier, crit_, sens_):
    cfr, cfa = CFR[crit_], c_fa(sens_)
    return np.where(tier == 0, np.where(y_ == 1, cfa, 0.0),
           np.where(tier == 1, RESTRICT_F * np.where(y_ == 1, cfa, cfr),
                               np.where(y_ == 0, cfr, 0.0)))

def metrics(y_, intervene, score=None):
    y_ = np.asarray(y_); d = np.asarray(intervene).astype(bool)
    tp, fp = ((y_ == 1) & d).sum(), ((y_ == 0) & d).sum()
    fn, tn = ((y_ == 1) & ~d).sum(), ((y_ == 0) & ~d).sum()
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    out = dict(accuracy=(tp + tn) / len(y_), precision=prec, recall=rec,
               f1=2 * prec * rec / max(prec + rec, 1e-9), FAR=fn / max(fn + tp, 1), FRR=fp / max(fp + tn, 1))
    if score is not None:
        out["roc_auc"] = roc_auc_score(y_, score); out["pr_auc"] = average_precision_score(y_, score)
    return out

def wilson(k, n_, z=1.96):
    if n_ == 0: return (np.nan, np.nan)
    p = k / n_; den = 1 + z * z / n_
    c = (p + z * z / (2 * n_)) / den
    h = z * np.sqrt(p * (1 - p) / n_ + z * z / (4 * n_ * n_)) / den
    return c - h, c + h

## 7. Main experiment — fit on the past, evaluate on the untouched future

In [ ]:
t0 = time.perf_counter()
bundle = fit_pipeline(df, tr, cal)
ledger = new_seeded_ledger()                       # fresh ledger; revocations are committed to it while streaming
out = apply_pipeline(bundle, df, te, ledger=ledger)
print(f"fit + apply: {time.perf_counter() - t0:.1f}s | ledger blocks after revocations: {len(ledger.chain)}")

yt, ct, st = y[te], crit[te], sens[te]
gate_fail_te = out["H"]["gate"][te] != ""
preds = {}                                          # name -> dict(deny/tier, score)

# --- A0: traditional perimeter rule (per-device-type 99th percentile of normal traffic, tuned on train only) ---
norm_tr = df.iloc[tr][df.iloc[tr].label == 0]
PER = [c for c in ["src_bytes", "dst_bytes", "src_pkts", "req_per_min"] if c in df.columns]
thr = norm_tr.groupby("device_type")[PER].quantile(.99)
dte = df.iloc[te]
t_ = thr.loc[dte.device_type.values].values
perim = (dte[PER].values > t_).any(axis=1)
preds["A0 Perimeter rule"] = dict(deny=perim, score=perim.astype(float))

# --- strong ML baselines, network+context features only (what v1 should have compared against) ---
X1 = df[F1_COLS].values
preds["B1 Random Forest (network)"] = dict(deny=out["p_net"][te] >= .5, score=out["p_net"][te])
hgb = HistGradientBoostingClassifier(max_iter=200, random_state=SEED).fit(X1[tr], y[tr]); ph = hgb.predict_proba(X1[te])[:, 1]
preds["B2 HistGradientBoosting (network)"] = dict(deny=ph >= .5, score=ph)
LOGC = [i for i, c in enumerate(F1_COLS) if c in SKEW_SET]
def mlp_in(X):
    Z = X.copy().astype(float); Z[:, LOGC] = np.log1p(np.clip(Z[:, LOGC], 0, None)); return Z
sc = StandardScaler().fit(mlp_in(X1[tr]))
mlp = MLPClassifier((64, 32), early_stopping=True, max_iter=300, random_state=SEED).fit(sc.transform(mlp_in(X1[tr])), y[tr])
pm = mlp.predict_proba(sc.transform(mlp_in(X1[te])))[:, 1]
preds["B3 MLP (network)"] = dict(deny=pm >= .5, score=pm)

# --- stepwise ablation of the proposed framework ---
preds["A2 Ledger gate only (no ML)"] = dict(deny=gate_fail_te, score=gate_fail_te.astype(float))
p_a3 = np.where(gate_fail_te, 1.0, out["p_net"][te])
preds["A3 Gate + RF (no history)"] = dict(deny=p_a3 >= .5, score=p_a3)
# A3b isolates the effect of the history features: same stacked second-stage RF, but WITHOUT the ledger-history columns
k_ = len(F1_COLS) + 1                               # NET + CTX + p_net
trg_ = tr[out["ok"][tr]]
m3 = rf().fit(out["X2"][trg_][:, :k_], y[trg_])
p3 = np.ones(len(df)); ti_ = te[out["ok"][te]]; p3[ti_] = m3.predict_proba(out["X2"][ti_][:, :k_])[:, 1]
preds["A3b Gate + stacked RF (no history)"] = dict(deny=p3[te] >= .5, score=p3[te])
preds["A4 Gate + RF + ledger trust history"] = dict(deny=out["p2"][te] >= .5, score=out["p2"][te])
tier = policy_tier(out["pc"][te], ct, st)
preds["A5 Full: A4 + criticality-aware tiers"] = dict(deny=tier >= 1, score=out["pc"][te], tier=tier)

rows = []
for name, p in preds.items():
    tr_tier = p.get("tier", 2 * p["deny"].astype(int))
    m = metrics(yt, p["deny"], p["score"])
    m["FRR_hard_deny"] = ((yt == 0) & (tr_tier == 2)).sum() / max((yt == 0).sum(), 1)
    m["legit_restricted"] = ((yt == 0) & (tr_tier == 1)).sum() / max((yt == 0).sum(), 1)
    m["cost_per_session"] = session_cost(yt, tr_tier, ct, st).mean()
    m["lifecritical_legit_denied"] = ((yt == 0) & (ct == 4) & (tr_tier == 2)).sum() / max(((yt == 0) & (ct == 4)).sum(), 1)
    rows.append(dict(method=name, **m))
ablation = pd.DataFrame(rows).set_index("method")
ablation.to_csv(os.path.join(RES_DIR, "ablation_test.csv"))
show = ablation[["accuracy", "precision", "recall", "f1", "FAR", "FRR", "FRR_hard_deny", "legit_restricted",
                 "roc_auc", "pr_auc", "cost_per_session", "lifecritical_legit_denied"]]
pd.set_option("display.width", 220)
print(f"TEST period only: {len(te):,} sessions, {yt.sum():,} attacks ({yt.mean():.1%}); natural class imbalance kept\n")
print(show.round(4).to_string())
print("\nFor A5, 'FAR/recall/FRR' count ANY intervention (restrict or deny); FRR_hard_deny counts full denials only.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
names = list(ablation.index); ypos = np.arange(len(names))[::-1]
colors = [COL["base"] if not nm.startswith(("A3", "A4", "A5")) else COL["full"] for nm in names]
ax[0].barh(ypos, ablation["recall"] * 100, color=colors); ax[0].set_title("Attack detection rate (%)"); ax[0].set_xlim(0, 100)
ax[1].barh(ypos, ablation["FRR"] * 100, color=colors); ax[1].set_title("Legit sessions blocked / restricted (%)")
ax[2].barh(ypos, ablation["cost_per_session"], color=colors); ax[2].set_title("Expected cost per session")
ax[0].set_yticks(ypos); ax[0].set_yticklabels(names)
savefig("fig1_ablation")

### 7b. Leakage and generalisation check on the real flows — hold out whole source machines

TON_IoT attacks come from a handful of attacker machines, and normal traffic from others. A random split lets a model memorise *who* is talking.
Here every flow from ~30 % of the source IPs is held out (`orig_src_ip`, never used as a feature). Only real TON_IoT rows are used.
A large drop versus the random split means the headline ML numbers on TON_IoT are optimistic.

In [ ]:
if "orig_src_ip" in df.columns:
    real = df[df.source == "ton_iot_real"].reset_index(drop=True)
    ips = np.array(sorted(real.orig_src_ip.unique())); rng_ip = np.random.default_rng(SEED); rng_ip.shuffle(ips)
    held = set(ips[: int(.3 * len(ips))])
    is_te = real.orig_src_ip.isin(held).values
    Xr, yr = real[F1_COLS].values, real.label.values
    rows_ = []
    for name, mtr, mte in [("random split (same source IPs seen in training)", np.random.default_rng(SEED).random(len(real)) > .3, None),
                           ("hold-out source IPs (unseen machines)", ~is_te, is_te)]:
        mte = ~mtr if mte is None else mte
        m_ = rf().fit(Xr[mtr], yr[mtr]); p_ = m_.predict_proba(Xr[mte])[:, 1]
        yy_ = yr[mte]; r_ = metrics(yy_, p_ >= .5, p_)
        rows_.append(dict(split=name, n_test=int(mte.sum()), attacks=int(yy_.sum()), recall=r_["recall"], FRR=r_["FRR"], f1=r_["f1"], roc_auc=r_["roc_auc"]))
    leak = pd.DataFrame(rows_); leak.to_csv(os.path.join(RES_DIR, "src_ip_holdout_check.csv"), index=False)
    print(leak.round(4).to_string(index=False))
    print(f"\nsource IPs: {len(ips)} total, {len(held)} held out; attacker and normal machines are largely disjoint in TON_IoT.")
else:
    print("not applicable for the synthetic dataset")

## 8. Where does each approach fail? — per-attack detection with 95 % Wilson intervals

In [ ]:
atk = df.iloc[te].attack_type.values
rows = []
for a in sorted(set(atk) - {"Normal"}):
    m = atk == a; k = m.sum()
    r = dict(attack=a, n=k)
    for nm, key in [("ML-only RF (B1)", "B1 Random Forest (network)"), ("Gate+RF+history (A4)", "A4 Gate + RF + ledger trust history"),
                    ("Full tiers (A5)", "A5 Full: A4 + criticality-aware tiers")]:
        hit = preds[key]["deny"][m].sum(); lo, hi = wilson(hit, k)
        r[nm] = hit / k; r[nm + " lo"] = lo; r[nm + " hi"] = hi
    rows.append(r)
per_attack = pd.DataFrame(rows).set_index("attack")
per_attack.to_csv(os.path.join(RES_DIR, "per_attack_test.csv"))
print(per_attack[["n", "ML-only RF (B1)", "Gate+RF+history (A4)", "Full tiers (A5)"]].round(3).to_string())
print("\nRows with n < 100 have wide intervals - report the CI, not just the point estimate.")

fig, ax = plt.subplots(figsize=(8, 4.2))
ix = np.arange(len(per_attack)); w = .38
for j, (nm, c) in enumerate([("ML-only RF (B1)", COL["base"]), ("Gate+RF+history (A4)", COL["full"])]):
    v = per_attack[nm].values
    err = [np.clip(v - per_attack[nm + " lo"].values, 0, None), np.clip(per_attack[nm + " hi"].values - v, 0, None)]
    ax.barh(ix + (j - .5) * w, v * 100, w, xerr=np.array(err) * 100, color=c, label=nm, capsize=2)
ax.set_yticks(ix); ax.set_yticklabels([f"{a} (n={n_})" for a, n_ in zip(per_attack.index, per_attack.n)])
ax.set_xlabel("detection rate (%)"); ax.legend(loc="upper center", bbox_to_anchor=(.5, -.16), ncol=2, frameon=False); ax.set_xlim(0, 105)
savefig("fig2_per_attack")

### Cost by device criticality — why the tiered policy matters clinically

In [ ]:
rows = []
for c_ in [1, 2, 3, 4]:
    m = ct == c_
    for nm, key in [("A4 fixed 0.5", "A4 Gate + RF + ledger trust history"), ("A5 criticality-aware", "A5 Full: A4 + criticality-aware tiers")]:
        tt = preds[key].get("tier", 2 * preds[key]["deny"].astype(int))[m]
        rows.append(dict(criticality=c_, policy=nm, sessions=m.sum(),
                         legit_denied=((yt[m] == 0) & (tt == 2)).sum() / max((yt[m] == 0).sum(), 1),
                         legit_restricted=((yt[m] == 0) & (tt == 1)).sum() / max((yt[m] == 0).sum(), 1),
                         attacks_allowed=((yt[m] == 1) & (tt == 0)).sum() / max((yt[m] == 1).sum(), 1),
                         cost=session_cost(yt[m], tt, ct[m], st[m]).mean()))
by_crit = pd.DataFrame(rows); by_crit.to_csv(os.path.join(RES_DIR, "cost_by_criticality.csv"), index=False)
print(by_crit.round(4).to_string(index=False))
piv = by_crit.pivot(index="criticality", columns="policy", values="cost")
ax = piv.plot.bar(figsize=(6, 3.5), color=[COL["base"], COL["full"]], rot=0)
ax.set_ylabel("expected cost per session"); ax.set_xlabel("device criticality (4 = life-critical)")
savefig("fig3_cost_by_criticality")

## 9. Robustness across time — forward-chaining evaluation (3 folds, mean ± std)
Each fold trains on the past only and tests on the next window.

In [ ]:
def eval_fold(a, b):
    tr_f = np.arange(0, int(.8 * a * n)); cal_f = np.arange(int(.8 * a * n), int(a * n)); te_f = np.arange(int(a * n), int(b * n))
    bd = fit_pipeline(df, tr_f, cal_f); o = apply_pipeline(bd, df, te_f)
    yt_, ct_, st_ = y[te_f], crit[te_f], sens[te_f]; gf = o["H"]["gate"][te_f] != ""
    res = {}
    for nm, deny, tier_ in [("B1 RF (network)", o["p_net"][te_f] >= .5, None),
                            ("A3 Gate + RF", gf | (o["p_net"][te_f] >= .5), None),
                            ("A4 Gate + RF + history", o["p2"][te_f] >= .5, None),
                            ("A5 Full tiers", None, policy_tier(o["pc"][te_f], ct_, st_))]:
        t_ = tier_ if tier_ is not None else 2 * deny.astype(int)
        m = metrics(yt_, t_ >= 1); m["cost"] = session_cost(yt_, t_, ct_, st_).mean()
        m["FRR_hard_deny"] = ((yt_ == 0) & (t_ == 2)).sum() / max((yt_ == 0).sum(), 1)
        res[nm] = m
    return res

t0 = time.perf_counter(); fold_rows = []
for i, (a, b) in enumerate([(.55, .70), (.70, .85), (.85, 1.0)]):
    for nm, m in eval_fold(a, b).items():
        fold_rows.append(dict(fold=i + 1, method=nm, **m))
folds = pd.DataFrame(fold_rows); folds.to_csv(os.path.join(RES_DIR, "forward_chaining_folds.csv"), index=False)
summ = folds.groupby("method")[["f1", "recall", "FAR", "FRR", "FRR_hard_deny", "cost"]].agg(["mean", "std"]).round(4)
print(f"{time.perf_counter() - t0:.0f}s"); print(summ.to_string())

## 10. Adversarial evasion

A compromised device can try to *shrink* its network footprint towards its device-type's normal profile
(ε = 0 no change, ε = 1 the flow is throttled to the size of a median normal flow). We perturb the test-period attacks and measure detection.
Identity attacks are not perturbed (their traffic is already normal-looking) — they are stopped by the gate, not by the model.

**Control test:** the same transform is also applied to *legitimate* flows. A model that learned attack semantics should not start flagging them; one that merely learned the fingerprint of the transform will.
**Expect a real drop for network/data attacks at high ε; report it honestly.**

In [ ]:
MED = compute_med(df[F1_COLS].values, y, tr)

def evade(eps, target="attacks"):
    # target="attacks": network/data attacks throttle their footprint (the real threat).
    # target="legit": CONTROL - the SAME transform applied to legitimate flows. A model that learned attack semantics must not start
    # flagging them; one that learned the transform's fingerprint will.
    d2 = df.copy(); m = np.zeros(len(df), bool)
    m[te] = np.isin(df.attack_family.values[te], ["network", "data"]) if target == "attacks" else (y[te] == 0)
    Xp = perturb_X1(df[F1_COLS].values[m], eps, MED)
    for j, c in enumerate(F1_COLS):
        if c in LOGCOLS or c in INT_COLS or c == "data_volume_kb":
            d2[c] = d2[c].astype(float); d2.loc[m, c] = Xp[:, j]
    return d2

fam = df.attack_family.values
t0 = time.perf_counter(); bundle_rob = fit_pipeline(df, tr, cal, robust=True); print(f"evasion-aware fit: {time.perf_counter() - t0:.0f}s")
SYS = [("B1 RF, standard", bundle, "p_net"), ("A4 full, standard", bundle, "p2"),
       ("B1r RF, evasion-aware", bundle_rob, "p_net"), ("A4r full, evasion-aware", bundle_rob, "p2")]
rows = []
for eps in [0, .25, .5, .75, 1.0]:
    d2, d2c = evade(eps), evade(eps, "legit")
    for nm, bd, key in SYS:
        oc = apply_pipeline(bd, d2c, te); rows.append(dict(eps=eps, method=nm, group="CONTROL: perturbed legit sessions", rate=(oc[key][te] >= .5)[yt == 0].mean()))
        o = apply_pipeline(bd, d2, te); deny = o[key][te] >= .5
        for grp, mask in [("network+data attacks", np.isin(fam[te], ["network", "data"])), ("identity attacks", fam[te] == "identity"),
                          ("all attacks", yt == 1), ("legit sessions (FRR)", yt == 0)]:
            rows.append(dict(eps=eps, method=nm, group=grp, rate=deny[mask].mean()))
evas = pd.DataFrame(rows); evas.to_csv(os.path.join(RES_DIR, "evasion.csv"), index=False)
print(evas.pivot_table(index=["group", "method"], columns="eps", values="rate").round(3).to_string())

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
styles = {"B1 RF, standard": (COL["base"], "--"), "A4 full, standard": (COL["full"], "--"),
          "B1r RF, evasion-aware": (COL["base"], "-"), "A4r full, evasion-aware": (COL["full"], "-")}
for a_, grp in zip(ax, ["network+data attacks", "all attacks", "CONTROL: perturbed legit sessions"]):
    for nm, (c, ls) in styles.items():
        s_ = evas[(evas.group == grp) & (evas.method == nm)]
        a_.plot(s_.eps, s_.rate * 100, "o" + ls, color=c, label=nm)
    a_.set_title(grp); a_.set_xlabel("evasion strength ε")
ax[0].set_ylabel("rate (%)"); ax[0].legend(fontsize=7)
savefig("fig4_evasion")

## 11. Privacy-aware training — per-hospital models, one-shot federated ensemble

Each hospital trains a stage-2 model on **its own** sessions only; the coordinator averages the models' *probabilities*
(weighted by sample count). Raw sessions never leave a hospital, and each model's SHA-256 hash and sample count is committed to the
ledger as a `MODEL_UPDATE` record. Hospitals are non-IID (different device mixes and attack mixes).

*Limits:* this is model-level federation (ensemble), not gradient averaging; there is **no formal differential-privacy
guarantee**, and tree models can leak. Call it "data-local training with ledger-audited model provenance", not "private by proof".

In [ ]:
X2, ok = out["X2"], out["ok"]
local, nloc = {}, {}
for h in range(5):
    idx = tr[(hosp[tr] == h) & ok[tr]]
    local[h] = rf(80).fit(X2[idx], y[idx]); nloc[h] = len(idx)
    ledger.append([{"type": "MODEL_UPDATE", "hospital": h, "n_samples": int(len(idx)),
                    "model_sha256": sha(pickle.dumps(local[h]))[:32], "round": 1}], ts=int(df.timestamp.values[tr[-1]]))

tg = te[ok[te]]; cg = cal[ok[cal]]
p_central = bundle["m2"].predict_proba(X2[tg])[:, 1]
P = np.column_stack([local[h].predict_proba(X2[tg])[:, 1] for h in range(5)])
Pc = np.column_stack([local[h].predict_proba(X2[cg])[:, 1] for h in range(5)])
w = np.array([nloc[h] for h in range(5)], float); wn = w / w.sum()
p_fed, p_fed_cal = P @ wn, Pc @ wn

# Per-site calibration: each hospital fits a 1-D isotonic map of the federated score on ITS OWN calibration window (only the map is
# used; no data leaves the site). Sites with too few positives or negatives in that window fall back to the raw score.
fed_site, used = p_fed.copy(), {}
for h in range(5):
    mc, mt = hosp[cg] == h, hosp[tg] == h
    pos, neg = int(y[cg][mc].sum()), int((1 - y[cg][mc]).sum())
    used[h] = pos >= 20 and neg >= 20
    if used[h]:
        fed_site[mt] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p_fed_cal[mc], y[cg][mc]).predict(p_fed[mt])
rows = []
for h in range(5):
    m = hosp[tg] == h
    f1 = lambda p: metrics(y[tg][m], p[m] >= .5)["f1"]
    rows.append(dict(hospital=h, n_test=int(m.sum()), n_train=nloc[h], site_calibrated=used[h],
                     AUC_local_only=roc_auc_score(y[tg][m], P[m, h]), AUC_federated=roc_auc_score(y[tg][m], p_fed[m]),
                     AUC_centralised=roc_auc_score(y[tg][m], p_central[m]),
                     F1_local_only=metrics(y[tg][m], P[m, h] >= .5)["f1"], F1_federated_raw=f1(p_fed),
                     F1_federated_site_cal=f1(fed_site), F1_centralised=f1(p_central)))
fed = pd.DataFrame(rows).set_index("hospital"); fed.to_csv(os.path.join(RES_DIR, "federated.csv"))
print(fed.round(4).to_string())
print("\nmean AUC  local-only %.4f | federated %.4f | centralised %.4f" % tuple(fed[c].mean() for c in ["AUC_local_only", "AUC_federated", "AUC_centralised"]))
print("mean F1   local-only %.4f | federated raw %.4f | federated site-calibrated %.4f | centralised %.4f" %
      tuple(fed[c].mean() for c in ["F1_local_only", "F1_federated_raw", "F1_federated_site_cal", "F1_centralised"]))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
fed[["AUC_local_only", "AUC_federated", "AUC_centralised"]].plot.bar(ax=ax[0], color=[COL["base"], COL["accent"], COL["full"]], rot=0)
ax[0].set_ylim(max(.5, fed[["AUC_local_only", "AUC_federated", "AUC_centralised"]].values.min() - .03), 1.005); ax[0].set_ylabel("ROC-AUC")
fed[["F1_local_only", "F1_federated_raw", "F1_federated_site_cal", "F1_centralised"]].plot.bar(ax=ax[1], color=[COL["base"], COL["accent"], COL["good"], COL["full"]], rot=0)
ax[1].set_ylabel("F1 at threshold 0.5"); ax[1].legend(fontsize=7)
savefig("fig5_federated")

## 12. Explainable, auditable decisions anchored on the ledger

Every test-period decision is written to the ledger with a **reason code** (gate rule, or the model's top-3 contributing features:
global importance × |z-score vs. that device-type's normal traffic|) and the **SHA-256 of that explanation**.
Device IDs are stored hashed. An auditor can later recompute the explanation, hash it and compare it against the
ledger, then prove inclusion with a Merkle proof.

In [ ]:
imp = dict(zip(F1_COLS, bundle["m_net"].feature_importances_))
EXPL = [c for c in NET if not c.endswith(("_code", "_flag")) and not c.startswith("has_") and c != "dst_port"]   # z-scores only make sense for magnitudes
net_imp = np.array([imp[c] for c in EXPL])
def tfm(frame):
    Z = frame[EXPL].copy().astype(float)
    for c in EXPL:
        if c in SKEW_SET: Z[c] = np.log1p(Z[c].clip(lower=0))
    return Z
Ztr = tfm(norm_tr); mu = Ztr.groupby(norm_tr.device_type).mean(); sd = Ztr.groupby(norm_tr.device_type).std().replace(0, 1).fillna(1)
Zte = tfm(dte)
zs = (Zte.values - mu.loc[dte.device_type.values].values) / sd.loc[dte.device_type.values].values
contrib = np.abs(zs) * net_imp

model_hash = sha(pickle.dumps(bundle["m2"]))[:16]
tiers_te = preds["A5 Full: A4 + criticality-aware tiers"]["tier"]
TIER = ["ALLOW", "RESTRICT", "DENY"]
gate_te = out["H"]["gate"][te]; pc_te = out["pc"][te]
txs, explanations = [], []
for i in range(len(te)):
    if gate_te[i]:
        ex = {"reason": gate_te[i]}
    else:
        top = np.argsort(-contrib[i])[:3]
        ex = {"reason": "MODEL", "top": [[EXPL[j], round(float(zs[i, j]), 2)] for j in top], "p_attack": round(float(pc_te[i]), 4)}
    explanations.append(ex)
    txs.append({"type": "ACCESS", "sid": df.session_id.values[te[i]], "did_h": sha(df.claimed_did.values[te[i]].encode())[:16],
                "tier": TIER[tiers_te[i]], "p": round(float(pc_te[i]), 4), "reason": ex["reason"],
                "expl_sha": sha(json.dumps(ex, sort_keys=True).encode())[:16], "model": model_hash})
t0 = time.perf_counter()
for i in range(0, len(txs), 100):
    ledger.append(txs[i:i + 100], ts=int(df.timestamp.values[te[i]]))
print(f"{len(txs):,} access decisions logged in {time.perf_counter() - t0:.2f}s -> chain length {len(ledger.chain)} blocks")
t0 = time.perf_counter(); ok_, bad, why = ledger.verify(); print(f"full chain verification: {ok_} ({why}) in {time.perf_counter() - t0:.2f}s")

# --- audit one denied life-critical decision end to end ---
cand = [i for i in range(len(te)) if tiers_te[i] == 2 and ct[i] == 4]
i = cand[0] if cand else int(np.argmax(tiers_te == 2))
blk_i, tx_i = next((b["index"], j) for b in ledger.chain for j, t in enumerate(b["txs"]) if t.get("sid") == df.session_id.values[te[i]])
print("\nExample audited decision:", json.dumps(ledger.chain[blk_i]["txs"][tx_i]))
print("Explanation recomputed from data:", explanations[i])
print("hash matches ledger:", sha(json.dumps(explanations[i], sort_keys=True).encode())[:16] == ledger.chain[blk_i]["txs"][tx_i]["expl_sha"],
      "| Merkle/quorum audit:", ledger.audit_tx(blk_i, tx_i))

### Tamper and consensus tests — what the ledger adds that a plain database does not

In [ ]:
tests = []
def attempt(name, fn):
    L = copy.deepcopy(ledger); expect_detect = fn(L)
    ok_, bad, why = L.verify()
    tests.append(dict(test=name, detected=(not ok_) if expect_detect is None else expect_detect, detail=why if not ok_ else "chain still verifies", block=bad))

k = next(b["index"] for b in ledger.chain if b["txs"] and b["txs"][0].get("type") == "ACCESS") + 5
def t1(L):   # flip a DENY into ALLOW in place
    j = next(j for j, t in enumerate(L.chain[k]["txs"]) if t["tier"] == "DENY") if any(t["tier"] == "DENY" for t in L.chain[k]["txs"]) else 0
    L.chain[k]["txs"][j]["tier"] = "ALLOW"
def t2(L):   # attacker rewrites tx AND recomputes hashes, root, header, but has no validator keys
    b = L.chain[k]; b["txs"][0]["tier"] = "ALLOW"
    b["tx_hashes"] = [tx_hash(t) for t in b["txs"]]; b["root"] = merkle_root(b["tx_hashes"])
    b["hash"] = L._hdr(b["index"], b["ts"], b["prev"], b["root"], b["proposer"])
def t3(L):   # delete a block
    del L.chain[k]
def t4(L):   # one compromised validator re-signs a rewritten block (1 valid sig < quorum 3)
    b = L.chain[k]; b["txs"][0]["tier"] = "ALLOW"
    b["tx_hashes"] = [tx_hash(t) for t in b["txs"]]; b["root"] = merkle_root(b["tx_hashes"])
    b["hash"] = L._hdr(b["index"], b["ts"], b["prev"], b["root"], b["proposer"]); b["sigs"] = {0: L._sign(0, b["hash"])}
attempt("T1 flip a DENY to ALLOW in place", t1)
attempt("T2 rewrite block + recompute hashes (no validator keys)", t2)
attempt("T3 delete a block", t3)
attempt("T4 one compromised validator re-signs rewritten block", t4)

# rogue registry write: tries to register an attacker DID
before = len(ledger.state["registry"])
for n_sign in [1, 2, 3]:
    L = copy.deepcopy(ledger)
    r = L.append([{"type": "REGISTER", "did": "ROGUE-" + str(n_sign), "fw": "x", "device_type": "ventilator"}], ts=0, signers=range(n_sign))
    tests.append(dict(test=f"T5 rogue REGISTER signed by {n_sign} validator(s)", detected=r is None,
                      detail="rejected: no quorum" if r is None else "ACCEPTED (>= quorum colluding validators; trust assumption f < n/3)",
                      block=None))
tamper = pd.DataFrame(tests); tamper.to_csv(os.path.join(RES_DIR, "ledger_tamper_tests.csv"), index=False)
print(tamper.to_string(index=False))

In [ ]:
# Verification cost: full scan grows linearly; Merkle audit of one decision stays ~constant
rows = []
for n_tx in [2_000, 10_000, 40_000]:
    L = PermissionedLedger()
    base_tx = {"type": "ACCESS", "sid": "S", "did_h": "d" * 16, "tier": "ALLOW", "p": 0.01, "reason": "MODEL", "expl_sha": "e" * 16, "model": "m"}
    for i in range(0, n_tx, 100):
        L.append([dict(base_tx, sid=f"S{i + j}") for j in range(100)], ts=i)
    t0 = time.perf_counter(); L.verify(); full = time.perf_counter() - t0
    t0 = time.perf_counter()
    for _ in range(50): L.audit_tx(len(L.chain) // 2, 37)
    one = (time.perf_counter() - t0) / 50
    rows.append(dict(transactions=n_tx, blocks=len(L.chain), full_verify_s=full, audit_one_decision_ms=one * 1e3))
scal = pd.DataFrame(rows); scal.to_csv(os.path.join(RES_DIR, "ledger_scaling.csv"), index=False)
print(scal.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(scal.transactions, scal.full_verify_s * 1e3, "o-", color=COL["bad"], label="full-chain verification")
ax.plot(scal.transactions, scal.audit_one_decision_ms, "o-", color=COL["good"], label="Merkle audit of one decision")
ax.set_yscale("log"); ax.set_xlabel("transactions on ledger"); ax.set_ylabel("time (ms, log)"); ax.legend()
savefig("fig6_ledger_scaling")

## 13. Secure channel — X25519 → HKDF → AES-256-GCM, with round-trip and tamper tests

Only sessions that pass the policy get a channel. A per-session key is derived by ECDH (not a fresh random key that nobody can decrypt),
the DID and nonce are bound as authenticated data, and decryption is actually tested.

In [ ]:
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag

def derive_key(priv, peer_pub, info=b"bzt-shis-v2"):
    return HKDF(hashes.SHA256(), 32, None, info).derive(priv.exchange(peer_pub))

dev_priv, gw_priv = X25519PrivateKey.generate(), X25519PrivateKey.generate()
k_dev, k_gw = derive_key(dev_priv, gw_priv.public_key()), derive_key(gw_priv, dev_priv.public_key())
assert k_dev == k_gw, "both ends must derive the same key"

rng_ = np.random.default_rng(SEED)
def vitals():
    return json.dumps({"hr": int(rng_.normal(75, 8)), "spo2": int(np.clip(rng_.normal(97, 1.2), 80, 100)),
                       "sbp": int(rng_.normal(120, 10)), "dbp": int(rng_.normal(80, 7)), "ts": int(time.time())}).encode()

aes_d, aes_g = AESGCM(k_dev), AESGCM(k_gw)
did16 = sha(b"demo-device")[:16].encode()
enc_t, dec_t, sizes, ok_count = [], [], [], 0
for _ in range(2000):
    pt = vitals(); iv = os.urandom(12); aad = did16 + iv
    t0 = time.perf_counter(); ct_ = aes_d.encrypt(iv, pt, aad); enc_t.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); rt = aes_g.decrypt(iv, ct_, aad); dec_t.append(time.perf_counter() - t0)
    ok_count += rt == pt; sizes.append((len(pt), len(ct_) + 12 + len(did16)))
print(f"round-trip correct: {ok_count}/2000")
bad = bytearray(ct_); bad[3] ^= 1
try: aes_g.decrypt(iv, bytes(bad), aad); print("tamper NOT detected")
except InvalidTag: print("bit-flip in ciphertext -> InvalidTag (detected)")
try: aes_g.decrypt(iv, ct_, b"x" * 28); print("AAD swap NOT detected")
except InvalidTag: print("swapped DID/IV (AAD) -> InvalidTag (detected)")

sz = np.array(sizes); overhead = (sz[:, 1] - sz[:, 0]) / sz[:, 0]
t0 = time.perf_counter()
for _ in range(500): derive_key(X25519PrivateKey.generate(), gw_priv.public_key())
hs = (time.perf_counter() - t0) / 500
crypto = pd.DataFrame([dict(payload_bytes_mean=sz[:, 0].mean(), wire_bytes_mean=sz[:, 1].mean(),
                            overhead_bytes=(sz[:, 1] - sz[:, 0]).mean(), overhead_pct=overhead.mean() * 100,
                            encrypt_us_median=np.median(enc_t) * 1e6, encrypt_us_p95=np.percentile(enc_t, 95) * 1e6,
                            decrypt_us_median=np.median(dec_t) * 1e6, handshake_ms=hs * 1e3)])
crypto.to_csv(os.path.join(RES_DIR, "crypto_overhead.csv"), index=False); print(crypto.round(2).T.to_string(header=False))

In [ ]:
# Decision latency, measured two honest ways
m2 = bundle["m2"].set_params(n_jobs=1)
idx_s = np.random.default_rng(SEED).choice(tg, 300, replace=False)
lat = []
for i in idx_s:
    t0 = time.perf_counter(); m2.predict_proba(X2[i:i + 1]); lat.append(time.perf_counter() - t0)
t0 = time.perf_counter(); bundle["m2"].set_params(n_jobs=-1).predict_proba(X2[tg]); bt = time.perf_counter() - t0
gate_us = dt / len(df) * 1e6
latency = pd.DataFrame([dict(single_request_model_ms_median=np.median(lat) * 1e3, single_request_model_ms_p95=np.percentile(lat, 95) * 1e3,
                             gate_plus_history_us=gate_us, batch_throughput_sessions_per_s=len(tg) / bt)])
latency.to_csv(os.path.join(RES_DIR, "latency.csv"), index=False); print(latency.round(3).T.to_string(header=False))

## 14. Headline summary

In [ ]:
a = ablation
print("=" * 78)
print(f"BZT-SHIS v2 - TEST PERIOD RESULTS (dataset = {DATASET}; see data note)")
print("=" * 78)
for k_ in ["B1 Random Forest (network)", "B2 HistGradientBoosting (network)", "B3 MLP (network)",
           "A3 Gate + RF (no history)", "A3b Gate + stacked RF (no history)", "A4 Gate + RF + ledger trust history", "A5 Full: A4 + criticality-aware tiers"]:
    r = a.loc[k_]
    print(f"{k_:42s} recall {r.recall:6.1%} | FAR {r.FAR:6.1%} | FRR {r.FRR:6.1%} | F1 {r.f1:.3f} | cost/session {r.cost_per_session:5.2f}")
print("-" * 78)
b1, a5 = a.loc["B1 Random Forest (network)"], a.loc["A5 Full: A4 + criticality-aware tiers"]
a4 = a.loc["A4 Gate + RF + ledger trust history"]
print(f"Expected-cost reduction, A5 vs best ML-only baseline : {(1 - a5.cost_per_session / a.loc[['B1 Random Forest (network)','B2 HistGradientBoosting (network)','B3 MLP (network)'],'cost_per_session'].min()):.1%}")
print(f"Expected-cost reduction, A5 vs A4 (fixed 0.5 threshold): {(1 - a5.cost_per_session / a4.cost_per_session):.1%}")
print(f"Life-critical legit sessions fully denied: A4 {a4.lifecritical_legit_denied:.2%}  ->  A5 {a5.lifecritical_legit_denied:.2%}")
_t = tamper[~tamper.test.str.contains("3 validator")]   # the 3-validator write is the control: a full quorum is accepted by design
print("Ledger attacks detected/rejected:", int(_t.detected.sum()), "/", len(_t), "(control: a colluding quorum of 3 validators is accepted - the f < n/3 trust assumption)")
print("=" * 78)
print("Files written to ./results and ./figures")

## 15. Limitations to state in the paper

1. **Synthetic data.** The ML-only vs. full-framework gap is partly determined by how stealthy identity attacks are generated. Validate on real traffic with real identity metadata.
2. **Simulated ledger.** Consensus is simulated (HMAC "signatures"); no network partitions, latency or Byzantine behaviour beyond the tests shown.
3. **Cost parameters are assumptions**, not clinical measurements. Present a sensitivity analysis or elicit them from clinicians.
4. **Federated component** is a one-shot ensemble, not gradient-based FL, and carries no differential-privacy guarantee.
5. **Compromised devices with valid keys** are only caught by behaviour; the evasion study shows where detection degrades.
6. Per-class results with small *n* have wide confidence intervals — always report them.